In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


df = pd.read_excel("online_retail_ii.xlsx")


df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]


df['Date'] = df['InvoiceDate'].dt.date


daily_demand = (
    df.groupby('Date')['Quantity']
      .sum()
      .reset_index()
)


daily_demand.columns = ['Date', 'Daily_Quantity']


daily_demand['Date'] = pd.to_datetime(daily_demand['Date'])


full_range = pd.date_range(
    start=daily_demand['Date'].min(),
    end=daily_demand['Date'].max()
)

daily_demand = (
    daily_demand
    .set_index('Date')
    .reindex(full_range, fill_value=0)
    .rename_axis('Date')
    .reset_index()
)


plt.figure(figsize=(12,4))
plt.plot(daily_demand['Date'], daily_demand['Daily_Quantity'])
plt.title("Daily Demand Over Time")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()

daily_demand.head(), daily_demand.tail()

In [ ]:
daily_demand = daily_demand.sort_values('Date')


In [ ]:
split_index = int(len(daily_demand) * 0.8)

train = daily_demand.iloc[:split_index]
test = daily_demand.iloc[split_index:]


In [ ]:
len(train), len(test)


In [ ]:
window = 7
last_7_day_avg = train['Daily_Quantity'].tail(window).mean()

test = test.copy()
test.loc[:, 'Forecast'] = last_7_day_avg

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.plot(train['Date'], train['Daily_Quantity'], label='Train')
plt.plot(test['Date'], test['Daily_Quantity'], label='Actual')
plt.plot(test['Date'], test['Forecast'], label='Forecast', linestyle='--')

plt.legend()
plt.title("Forecast vs Actual")
plt.xlabel("Date")
plt.ylabel("Daily Quantity")
plt.tight_layout()
plt.show()


In [ ]:
test['Error'] = test['Daily_Quantity'] - test['Forecast']
test['Abs_Error'] = test['Error'].abs()
test['APE'] = test['Abs_Error'] / test['Daily_Quantity'].replace(0, 1)


In [ ]:
mae = test['Abs_Error'].mean()
mape = test['APE'].mean()

mae, mape


In [ ]:
plt.figure(figsize=(12,4))
plt.plot(test['Date'], test['Abs_Error'])
plt.title("Absolute Forecast Error Over Time")
plt.xlabel("Date")
plt.ylabel("Absolute Error")
plt.tight_layout()
plt.show()


In [ ]:
test.head()


In [ ]:
test['Rolling_Abs_Error_7d'] = (
    test['Abs_Error']
    .rolling(window=7, min_periods=1)
    .mean()
)

test['Rolling_MAPE_7d'] = (
    test['APE']
    .rolling(window=7, min_periods=1)
    .mean()
)


In [ ]:
def assign_alert(mape):
    if mape > 0.40:
        return "HIGH"
    elif mape > 0.30:
        return "MEDIUM"
    elif mape > 0.20:
        return "LOW"
    else:
        return "OK"

test['Alert_Level'] = test['Rolling_MAPE_7d'].apply(assign_alert)

In [ ]:
alert_log = test[test['Alert_Level'] != 'OK'][[
    'Date',
    'Daily_Quantity',
    'Forecast',
    'Rolling_MAPE_7d',
    'Alert_Level'
]]


In [ ]:
alert_log.head()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,4))
plt.plot(test['Date'], test['Rolling_MAPE_7d'], label='Rolling MAPE')

plt.axhline(0.20, color='green', linestyle='--', label='LOW Threshold')
plt.axhline(0.30, color='orange', linestyle='--', label='MEDIUM Threshold')
plt.axhline(0.40, color='red', linestyle='--', label='HIGH Threshold')

plt.legend()
plt.title("Rolling Forecast Error & Alert Thresholds")
plt.xlabel("Date")
plt.ylabel("Rolling MAPE")
plt.tight_layout()
plt.show()


In [ ]:
test['Alert_Level'].value_counts()


In [ ]:
def recommend_action(alert):
    if alert == "HIGH":
        return "Immediate manual review and forecast override"
    elif alert == "MEDIUM":
        return "Monitor closely and prepare intervention"
    elif alert == "LOW":
        return "Observe, no immediate action"
    else:
        return "No action needed"

In [ ]:
test['Recommended_Action'] = test['Alert_Level'].apply(recommend_action)


In [ ]:
test[['Date', 'Alert_Level', 'Recommended_Action']].head()


In [ ]:
final_alert_table = test[test['Alert_Level'] != 'OK'][[
    'Date',
    'Daily_Quantity',
    'Forecast',
    'Rolling_MAPE_7d',
    'Alert_Level',
    'Recommended_Action'
]]

In [ ]:
final_alert_table.head()

Business Summary

- Built a demand forecasting system with continuous error monitoring
- Implemented rolling KPI-based early warning alerts
- Classified alerts into LOW, MEDIUM, HIGH severity
- Attached action recommendations for operations teams
- Goal: detect demand risk early and trigger timely human intervention

In [ ]:
final_alert_table.to_csv("alert_output.csv", index=False)

In [ ]:
final_alert_table.to_csv("alert_output.csv", index=False)